# 1. Prepare SfM subset once — CPU
Run this notebook once with the original SfM + Oxford + Paris datasets and Caffe weights attached. No checkpoint or GPU is needed. It selects the same subset, removes self-pairs, validates images, then copies the exact original SfM image bytes into a portable subset folder. Images are not recompressed or resized. The final manifest is written only after export succeeds.

Save the output folder `b0_sfm100_prepared` as a Kaggle Dataset, then attach that dataset to the training notebook. Keep Oxford/Paris and Caffe weights attached separately. This preparation run is intentionally expensive; training no longer performs it.


In [ ]:
# Session logging starts BEFORE dependency imports and data discovery.
import sys, time, threading, traceback, re
from pathlib import Path
from datetime import datetime

# Safe to rerun this cell: close the previous logger and remove its hooks.
if '_session_log_stop' in globals():
    _session_log_stop()

_session_dir = Path('/kaggle/working/b0_sfm100_notebook/logs')
_session_dir.mkdir(parents=True, exist_ok=True)
SESSION_LOG = _session_dir / ('session_' + str(time.time_ns()) + '.log')
_session_file = SESSION_LOG.open('a', buffering=65536, encoding='utf-8')
_session_stdout, _session_stderr = sys.stdout, sys.stderr
_session_lock = threading.RLock()
_session_event = threading.Event()
_session_state = {'stage': 'startup', 'since': time.monotonic(), 'active': True}

# Raw writes go to disk; only complete, filtered lines reach the notebook UI.
CONSOLE_PROGRESS_SECONDS = 60
class SessionTee:
    def __init__(self, console):
        self.console = console
        self.pending = ''
        self.last_progress = float('-inf')
    def _line(self, line):
        line = line.strip('\r')
        if not line.strip():
            return
        progress = re.match(r'^(?:>>>> \d+/\d+ done|SfM paths:|Image decode preflight:|>> (?:Train|Val):)', line)
        now = time.monotonic()
        if progress:
            # Keep only the latest rate-limited progress in the UI. Full log is on disk.
            if now - self.last_progress < CONSOLE_PROGRESS_SECONDS:
                return
            self.last_progress = now
        self.console.write(line + '\n')
    def write(self, value):
        with _session_lock:
            _session_file.write(value)
            parts = re.split(r'[\r\n]', self.pending + value)
            self.pending = parts.pop()
            for line in parts:
                self._line(line)
        return len(value)
    def flush(self):
        with _session_lock:
            # A flush must not turn each print fragment into a notebook output.
            self.console.flush()
            _session_file.flush()
    def finish(self):
        with _session_lock:
            if self.pending:
                self._line(self.pending)
                self.pending = ''
            self.flush()
    def __getattr__(self, name): return getattr(self.console, name)

sys.stdout, sys.stderr = SessionTee(_session_stdout), SessionTee(_session_stderr)

def stage(message):
    _session_state.update(stage=message, since=time.monotonic(), active=True)
    print(f'[{datetime.now().isoformat(timespec="seconds")}] {message}', flush=True)

def _session_heartbeat():
    while not _session_event.wait(60):
        if _session_state['active']:
            elapsed = time.monotonic() - _session_state['since']
            print(f'[heartbeat] {_session_state["stage"]} | {elapsed:.0f}s since stage start '
                  '(kernel alive; this does not prove GPU progress)', flush=True)

def _session_pre_cell(info):
    stage('Cell started: ' + info.raw_cell.strip().splitlines()[0][:120])

def _session_post_cell(result):
    error = result.error_before_exec or result.error_in_exec
    if error is not None:
        print('[CELL FAILED] Full traceback follows:', flush=True)
        traceback.print_exception(type(error), error, error.__traceback__)
    else:
        print('[CELL COMPLETE]', flush=True)
    sys.stdout.flush(); sys.stderr.flush()
    _session_state['active'] = False

_session_ip = get_ipython()
_session_ip.events.register('pre_run_cell', _session_pre_cell)
_session_ip.events.register('post_run_cell', _session_post_cell)
_session_thread = threading.Thread(target=_session_heartbeat, daemon=True)
_session_thread.start()

def _session_log_stop():
    _session_event.set()
    _session_thread.join(timeout=2)
    _session_ip.events.unregister('pre_run_cell', _session_pre_cell)
    _session_ip.events.unregister('post_run_cell', _session_post_cell)
    sys.stdout.finish(); sys.stderr.finish()
    sys.stdout, sys.stderr = _session_stdout, _session_stderr
    _session_file.close()

stage('Session log: ' + str(SESSION_LOG))
stage('Importing dependencies and checking GPUs')

import os, sys, json, pickle, random, math, time, hashlib, base64, zlib, re, gc
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from types import SimpleNamespace
import inspect, traceback
if sys.version_info < (3, 9):
    raise RuntimeError('Python >=3.9 is required for Path.is_relative_to and str.removeprefix.')
try:
    import numpy as np
    import PIL
    from PIL import Image
    import torch
    import torchvision
    from torchvision import transforms
except (ImportError, RuntimeError, OSError) as error:
    raise RuntimeError('Kaggle dependency import failed. Use a compatible preinstalled torch/torchvision pair; '
                       'do not install the historical repository requirements. Details: ' + str(error)) from error
required_apis = {
    'Pillow Image.Resampling.LANCZOS': hasattr(Image, 'Resampling') and hasattr(Image.Resampling, 'LANCZOS'),
    'NumPy isin': callable(getattr(np, 'isin', None)),
    'torch.load(weights_only=...)': 'weights_only' in inspect.signature(torch.load).parameters,
    'Optimizer.register_step_pre_hook': callable(getattr(torch.optim.Optimizer, 'register_step_pre_hook', None)),
    'torchvision VGG16 constructor': callable(getattr(torchvision.models, 'vgg16', None)),
    'torchvision image backend API': callable(getattr(torchvision, 'get_image_backend', None)),
}
missing = [name for name, present in required_apis.items() if not present]
if missing:
    raise RuntimeError('Missing required compatibility APIs: ' + ', '.join(missing))
torchvision.set_image_backend('PIL')
print('Dependency/API preflight passed:', required_apis)

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
if not INPUT_ROOT.is_dir() or not WORKING.is_dir():
    raise RuntimeError('Run on Kaggle with the three datasets attached and a GPU enabled.')
ENVIRONMENT = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__, numpy=np.__version__, pillow=PIL.__version__, preparation='CPU')
print('Free working disk GB:', __import__('shutil').disk_usage(WORKING).free / 1e9)
WORKSPACE = WORKING / 'b0_sfm100_notebook'
WORKSPACE.mkdir(exist_ok=True)

# Shared logger is defined before smoke; all run logs stay inside the notebook workspace.
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, value):
        for stream in self.streams:
            stream.write(value)
        return len(value)
    def flush(self):
        for stream in self.streams: stream.flush()

@contextmanager
def logged(path):
    with Path(path).open('a', buffering=65536) as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            yield


## 2. Offline repository setup (pinned source snapshot)
The notebook carries the original B0 Python source and license as compressed JSON. It materializes that exact snapshot under its own workspace, validates every SHA-256, and imports it without installation or Git operations. Existing differing files cause failure; unrelated files are never deleted. The source commit and per-file hashes are saved with run records. A fresh kernel is required if cirtorch was imported from a different checkout.


In [ ]:
SOURCE_COMMIT = '747166e627e514842f5c5be531f4e0f73612cc90'
SOURCE_HASHES = {'cirtorch/__init__.py': 'ef6a810989ab31e2d49446d7faaa9c735f85d2fccfdedafb531f41a85b5841b5', 'cirtorch/datasets/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/datasets/datahelpers.py': '6e18ba2c48ba66f000d83e94187f5a258862d4f490c73de1f94481da0ef811b3', 'cirtorch/datasets/genericdataset.py': '76bb2bab4493f5cfcda0691006442178727c7625abcc2e4e8ab40d6561d332d3', 'cirtorch/datasets/testdataset.py': '13c305c090657ca5624615c72ae2e8eaea26995632da9169d0b379728de117f3', 'cirtorch/datasets/traindataset.py': '2e79fa6a05ac4ee153fab383af438f6d092585205ba6ed8b1a86ab20266da702', 'cirtorch/examples/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/examples/example_descriptor_extraction.py': '6f408acfa97edee2da6d338ff59d8bab2828a079c7b9edd7a969b51dcc5aac8f', 'cirtorch/examples/test.py': '56bc6037a7e9e91873ae3f64aefeab46f983bee1966266e241346617b52b3f7c', 'cirtorch/examples/test_e2e.py': 'e7ec88802bfba68878df91fbfb617ba0b9d3e224f3689b24b554f24d4e380016', 'cirtorch/examples/train.py': '7f0cbfd34c8075139e6225ec7db72312b97c5a0f2d7762ace9c473c928f33848', 'cirtorch/layers/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/layers/functional.py': 'c016ffb064d458c920c61add3f6d03f1f11ae2e89d1e06facf5854636e41019c', 'cirtorch/layers/loss.py': 'fbaa3d82e297cb77481f93fe49de10af30158a0f94f487dc80bca5888f82715d', 'cirtorch/layers/normalization.py': '9ae607b73af5b0112c83a38db26586447e95f92359a3ca7eb7c5fefa3340f076', 'cirtorch/layers/pooling.py': 'c22d6d6b72e45d7f2454e6ea1b263b7ad45f548b8e50b1914fcce92bd74f8c64', 'cirtorch/networks/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/networks/imageretrievalnet.py': 'ffc64fca4c1077987f40640eec90500f7f26b80f3a37703432572f21e8288a60', 'cirtorch/utils/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/utils/download.py': 'd5bf645ec5417457b4062dbf010cc8388d3310d779cbe70beffd3e8000e2b195', 'cirtorch/utils/download_win.py': '643261a89fe1ceaaff19cb9c5244af178d4dc833542f87e51042a0f09e3aba19', 'cirtorch/utils/evaluate.py': 'ced1d110dd19f3f887e75f18ac9120aaff8900f5d6b52dc8c45c66c823141fd0', 'cirtorch/utils/general.py': '626e9f916199f7906926485c60b0a197e2ab112ab74cc66393ed5e54a295ea21', 'cirtorch/utils/whiten.py': '05162c86e6d8deea75d0d814d5469fa660e3006d2c98f6193a441dd08fb50dc1', 'LICENSE': 'c31c66c3945689e8876c9bbdb360228d13e4a15d84a77452dbe2db0ab67558e7'}
SOURCE_PAYLOAD = '...'
REPO_ROOT = WORKSPACE / ('source_' + SOURCE_COMMIT[:12])
source_files = json.loads(zlib.decompress(base64.b64decode(SOURCE_PAYLOAD)))
assert set(source_files) == set(SOURCE_HASHES)
for relative, contents in source_files.items():
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts or not (relative == 'LICENSE' or relative.startswith('cirtorch/')):
        raise RuntimeError('Unsafe embedded source path: ' + relative)
    encoded = contents.encode()
    assert hashlib.sha256(encoded).hexdigest() == SOURCE_HASHES[relative]
    dest = REPO_ROOT / rel
    if dest.exists():
        if dest.is_symlink() or dest.read_bytes() != encoded:
            raise RuntimeError('Existing source differs: ' + str(dest))
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(encoded)
if 'cirtorch' in sys.modules:
    imported = Path(sys.modules['cirtorch'].__file__).resolve()
    if not imported.is_relative_to(REPO_ROOT.resolve()):
        raise RuntimeError('Restart kernel: cirtorch is already imported from ' + str(imported))
sys.path.insert(0, str(REPO_ROOT))
import cirtorch
from cirtorch.datasets import traindataset, datahelpers
from cirtorch.datasets.testdataset import configdataset
from cirtorch.networks import imageretrievalnet
from cirtorch.layers.loss import ContrastiveLoss
from cirtorch.layers.pooling import GeM
from cirtorch.utils import general
from cirtorch.utils.evaluate import compute_ap, compute_map
from cirtorch.examples import train as original_train
assert Path(cirtorch.__file__).resolve().is_relative_to(REPO_ROOT.resolve())
print('Pinned source:', SOURCE_COMMIT, cirtorch.__file__)


## 3. Discover mounted inputs
Metadata and weights are discovered from mounted files. Later image roots default to the corresponding metadata folder and honor PATH_OVERRIDES. The existing subset-generation and self-pair cleanup steps follow below.


In [ ]:
# Set exact FILE paths here if bounded discovery reports missing/ambiguous files.
# Image directories default to metadata_parent/ims and ground_truth_parent/jpg.
PATH_OVERRIDES = dict(sfm_metadata=None, sfm_images=None, oxford_ground_truth=None,
                      oxford_images=None, paris_ground_truth=None, paris_images=None, weights=None)

def input_path(p):
    p = Path(p).resolve()
    if not p.is_relative_to(INPUT_ROOT.resolve()) or not p.exists():
        raise RuntimeError('Expected existing mounted input: ' + str(p))
    return p

WANTED_FILES = {
    'sfm_metadata': 'retrieval-SfM-120k.pkl',
    'oxford_ground_truth': 'gnd_oxford5k.pkl',
    'paris_ground_truth': 'gnd_paris6k.pkl',
    'weights': 'imagenet-caffe-vgg16-features-d369c8e.pth',
}

def discover_metadata(root, wanted, max_depth=7, max_dirs=200, max_entries=4000, seconds=30):
    # Do not descend into image trees or CID's two-hex-digit directories.
    from collections import deque
    pending = deque([(Path(root), 0)])
    found = {name: [] for name in wanted}
    visited = entries = 0
    started = time.monotonic()
    truncated = False
    skip = {'ims', 'jpg', 'images', 'image', 'jpeg', 'png', '__pycache__', '.git'}
    while pending:
        if visited >= max_dirs or entries >= max_entries or time.monotonic()-started >= seconds:
            truncated = True
            break
        directory, depth = pending.popleft()
        visited += 1
        if visited == 1 or visited % 20 == 0:
            print(f'Discovery: {visited} directories, {entries} entries, current={directory}', flush=True)
        for name in wanted:
            candidate = directory / name
            if candidate.is_file():
                found[name].append(candidate)
                print('Found:', candidate, flush=True)
        # Depth limit deliberately avoids scanning the contents of deep folders.
        if depth >= max_depth:
            continue
        with os.scandir(directory) as listing:
            for entry in listing:
                entries += 1
                if entries >= max_entries or time.monotonic()-started >= seconds:
                    truncated = True
                    break
                if entry.name.lower() in skip or re.fullmatch(r'[0-9a-fA-F]{2}', entry.name):
                    continue
                if entry.is_dir(follow_symlinks=False):
                    pending.append((Path(entry.path), depth+1))
        if truncated:
            break
    print(f'Discovery finished: dirs={visited}, entries={entries}, '
          f'seconds={time.monotonic()-started:.1f}, budget_reached={truncated}', flush=True)
    return found, truncated

stage('Locating metadata/weights (bounded scan; image trees skipped)')
unresolved = [name for key, name in WANTED_FILES.items() if PATH_OVERRIDES[key] is None]
file_index, discovery_truncated = discover_metadata(INPUT_ROOT, unresolved) if unresolved else ({}, False)

def choose(key, candidates):
    override = PATH_OVERRIDES[key]
    if override is not None:
        p = input_path(override)
        expected_name = WANTED_FILES.get(key)
        if not p.is_file() or (expected_name and p.name != expected_name):
            raise RuntimeError(f'{key}: expected file named {expected_name}: {p}')
        return p
    candidates = sorted(set(input_path(p) for p in candidates))
    if discovery_truncated or len(candidates) != 1:
        raise RuntimeError(f'{key}: bounded discovery found {candidates}; budget_reached={discovery_truncated}. '
                           'Set exact PATH_OVERRIDES file paths above and rerun this cell. '
                           'Discovery skips image folders and searches only 7 directory levels.')
    return candidates[0]

SFM_PKL = choose('sfm_metadata', file_index.get('retrieval-SfM-120k.pkl', []))
assert SFM_PKL.name == 'retrieval-SfM-120k.pkl'
with SFM_PKL.open('rb') as f:
    SFM_DB = pickle.load(f)  # Trusted team-supplied mounted metadata only.
PROVENANCE_PATH = SFM_PKL.parent / 'subset_provenance_k100_seed42.txt'
FILTER_PATH = SFM_PKL.parent / 'needed_files.txt'
# Generated provenance/filter are written after subset selection below.
SOURCE_SFM_PKL = SFM_PKL
WEIGHTS = choose('weights', file_index.get('imagenet-caffe-vgg16-features-d369c8e.pth', []))
print('Active metadata:', SFM_PKL)
print('Full backup is ignored; whitening metadata is unused.')


In [ ]:
# Reuse already loaded metadata: no second recursive scan of /kaggle/input.
stage('Inspecting selected SfM metadata: ' + str(SOURCE_SFM_PKL))
for split in ['train', 'val']:
    d = SFM_DB[split]
    print(split, 'images:', len(d['cids']), 'clusters:', len(set(d['cluster'])),
          'pairs:', len(d['qidxs']), flush=True)


In [ ]:
import pickle
import random
from pathlib import Path

SEED = 42
NUM_CLUSTERS = 100

OUT_DIR = Path("/kaggle/working/b0_sfm100_notebook/generated_subset100")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# SFM_DB currently contains FULL metadata.
with SOURCE_SFM_PKL.open('rb') as f:
    full_db = pickle.load(f)

def make_subset(split_data, selected_clusters):
    cids = split_data["cids"]
    clusters = split_data["cluster"]
    qidxs = split_data["qidxs"]
    pidxs = split_data["pidxs"]

    selected = set(selected_clusters)

    kept_indices = [
        i for i, cluster in enumerate(clusters)
        if cluster in selected
    ]

    old_to_new = {
        old_idx: new_idx
        for new_idx, old_idx in enumerate(kept_indices)
    }

    result = dict(split_data)

    result["cids"] = [cids[i] for i in kept_indices]
    result["cluster"] = [clusters[i] for i in kept_indices]

    pairs = [
        (old_to_new[q], old_to_new[p])
        for q, p in zip(qidxs, pidxs)
        if q in old_to_new and p in old_to_new
    ]

    result["qidxs"] = [q for q, p in pairs]
    result["pidxs"] = [p for q, p in pairs]

    # Preserve per-image metadata if present.
    for key, value in split_data.items():
        if key in ("cids", "cluster", "qidxs", "pidxs"):
            continue
        if isinstance(value, (list, tuple)) and len(value) == len(cids):
            result[key] = [value[i] for i in kept_indices]

    return result

# Select training clusters.
train_clusters = sorted(set(full_db["train"]["cluster"]))
if len(train_clusters) < NUM_CLUSTERS:
    raise RuntimeError("Not enough training clusters")

rng = random.Random(SEED)
selected_train = rng.sample(train_clusters, NUM_CLUSTERS)

# Preserve the existing validation split when it already has 20 clusters.
val_clusters = sorted(set(full_db["val"]["cluster"]))
if len(val_clusters) == 20:
    selected_val = val_clusters
else:
    selected_val = rng.sample(val_clusters, 20)

subset_db = dict(full_db)
subset_db["train"] = make_subset(full_db["train"], selected_train)
subset_db["val"] = make_subset(full_db["val"], selected_val)

def summary(db, split):
    d = db[split]
    return (
        len(d["cids"]),
        len(set(d["cluster"])),
        len(d["qidxs"])
    )

print("Train:", summary(subset_db, "train"))
print("Val:  ", summary(subset_db, "val"))

EXPECTED = {
    "train": (15953, 100, 34339),
    "val": (739, 20, 181),
}

for split, expected in EXPECTED.items():
    actual = summary(subset_db, split)
    accepted_pairs = {expected[2]}
    if split == 'train':
        accepted_pairs.add(34337)  # Already-cleaned copy of the same subset.
    if actual[:2] != expected[:2] or actual[2] not in accepted_pairs:
        raise RuntimeError(
            f"{split}: generated {actual}, expected {expected}. "
            "Subset selection differs from the team's original script. "
            "Do not use this subset for B0 yet."
        )

# Save only after exact validation passes.
subset_path = OUT_DIR / "retrieval-SfM-120k.pkl"

with subset_path.open("wb") as f:
    pickle.dump(subset_db, f, protocol=pickle.HIGHEST_PROTOCOL)

all_cids = (
    subset_db["train"]["cids"]
    + subset_db["val"]["cids"]
)

needed = [
    f"{cid[-2:]}/{cid[-4:-2]}/{cid[-6:-4]}/{cid}"
    for cid in all_cids
]

filter_path = OUT_DIR / "needed_files.txt"
filter_path.write_text(
    "\n".join(sorted(set(needed))) + "\n"
)

provenance_path = OUT_DIR / "subset_provenance_k100_seed42.txt"
provenance_path.write_text(
    "num_clusters=100 seed=42\n"
    "train: 100 clusters, 15,953 images, 34,339 pairs\n"
    "val: 20 clusters, 739 images, 181 pairs\n"
    "total_needed_images=16692\n"
)

# Activate generated metadata in the current notebook session.
SFM_PKL = subset_path
SFM_DB = subset_db
FILTER_PATH = filter_path
PROVENANCE_PATH = provenance_path

print("Subset-100 created successfully!")
print("Active metadata:", SFM_PKL)
print("Active filter:", FILTER_PATH)
print("Active provenance:", PROVENANCE_PATH)

In [ ]:
for part in ["train", "val"]:
    d = SFM_DB[part]
    clusters = d["cluster"]

    self_pairs = []
    cross_cluster_pairs = []

    for idx, (q, p) in enumerate(zip(d["qidxs"], d["pidxs"])):
        if q == p:
            self_pairs.append((idx, q, p))

        if clusters[q] != clusters[p]:
            cross_cluster_pairs.append(
                (idx, q, p, clusters[q], clusters[p])
            )

    print(f"\n{part.upper()}")
    print("Self-pairs:", len(self_pairs))
    print("Cross-cluster pairs:", len(cross_cluster_pairs))
    print("First self-pairs:", self_pairs[:5])
    print("First cross-cluster pairs:", cross_cluster_pairs[:5])

In [ ]:
# Remove self-pairs from SfM subset-100
from pathlib import Path
import pickle

for part in ["train", "val"]:
    d = SFM_DB[part]

    valid_pairs = [
        (q, p)
        for q, p in zip(d["qidxs"], d["pidxs"])
        if q != p
    ]

    removed = len(d["qidxs"]) - len(valid_pairs)

    d["qidxs"] = [q for q, p in valid_pairs]
    d["pidxs"] = [p for q, p in valid_pairs]

    print(f"{part}: removed {removed} self-pairs")

# Save corrected metadata
with open(SFM_PKL, "wb") as f:
    pickle.dump(SFM_DB, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved:", SFM_PKL)

In [ ]:
# Derive provenance from the active, cleaned metadata; never edit mounted inputs.
lines = ['num_clusters=100 seed=42']
for part in ['train', 'val']:
    d = SFM_DB[part]
    lines.append(f"{part}: {len(set(d['cluster']))} clusters, {len(d['cids']):,} images, {len(d['qidxs']):,} pairs")
lines.append(f"total_needed_images={len(set(SFM_DB['train']['cids'] + SFM_DB['val']['cids']))}")
PROVENANCE_PATH.write_text('\n'.join(lines) + '\n')
print(PROVENANCE_PATH.read_text())


## 4. Validate the fixed subset-100 metadata
Checks exact user-specified counts, all paired indices, unique CIDs, query/positive cluster consistency, and disjoint train/validation CIDs and cluster IDs. Checks provenance seed/counts and the complete extraction filter against the active metadata. The preceding cells generate the seed-42 subset from the mounted active metadata and remove self-pairs in the working copy. No mounted input is modified. Counts plus a provenance label do not independently prove the historical draw; hashes and selected IDs identify the exact supplied subset for future comparison.


In [ ]:
EXPECTED_SUBSET = {
    'train': (15953, 100, 34337),
    'val': (739, 20, 181)
}
assert isinstance(SFM_DB, dict) and all(part in SFM_DB for part in EXPECTED_SUBSET)
SUBSET_SUMMARY = {}
for part, expected in EXPECTED_SUBSET.items():
    d = SFM_DB[part]
    assert all(k in d for k in ['cids', 'cluster', 'qidxs', 'pidxs']), part
    cids, clusters, qs, ps = (d[k] for k in ['cids', 'cluster', 'qidxs', 'pidxs'])
    assert len(cids) == len(clusters) and len(qs) == len(ps), part
    actual = (len(cids), len(set(clusters)), len(qs))
    assert actual == expected, f'{part}: expected {expected}, got {actual}; refuse a different subset'
    assert len(set(cids)) == len(cids)
    assert all(isinstance(cid, str) and len(cid) >= 6 and Path(cid).name == cid
               and cid not in {'.', '..'} for cid in cids)
    assert all(isinstance(i, (int, np.integer)) and 0 <= i < len(cids) for i in list(qs) + list(ps))
    assert all(q != p and clusters[q] == clusters[p] for q, p in zip(qs, ps)), part
    SUBSET_SUMMARY[part] = dict(images=actual[0], clusters=actual[1], pairs=actual[2],
                                cluster_ids=sorted(int(c) if isinstance(c, np.integer) else c for c in set(clusters)))
assert not set(SFM_DB['train']['cids']) & set(SFM_DB['val']['cids']), 'CID leakage'
assert not set(SFM_DB['train']['cluster']) & set(SFM_DB['val']['cluster']), 'Cluster leakage'
ALL_CIDS = list(SFM_DB['train']['cids']) + list(SFM_DB['val']['cids'])
assert len(ALL_CIDS) == len(set(ALL_CIDS)) == 16692
provenance_text = PROVENANCE_PATH.read_text()
print(provenance_text)
assert re.search(r'num_clusters\s*=\s*100\b', provenance_text)
assert re.search(r'seed\s*=\s*42\b', provenance_text)
for part, (ni, nc, npairs) in EXPECTED_SUBSET.items():
    assert re.search(rf'{part}:\s*{nc}\s+clusters,\s*{ni:,}\s+images,\s*{npairs:,}\s+pairs', provenance_text)
expected_filter = {f'{cid[-2:]}/{cid[-4:-2]}/{cid[-6:-4]}/{cid}' for cid in ALL_CIDS}
actual_filter = {line.removeprefix('./') for line in FILTER_PATH.read_text().splitlines() if line.strip()}
assert actual_filter == expected_filter, 'needed_files.txt differs from active metadata'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

DATASET_IDENTITY = dict(subset_seed=42, partitions=SUBSET_SUMMARY,
                       metadata_sha256=sha256_file(SFM_PKL), provenance_sha256=sha256_file(PROVENANCE_PATH),
                       filter_sha256=sha256_file(FILTER_PATH), weights_sha256=sha256_file(WEIGHTS))
print(json.dumps(SUBSET_SUMMARY, indent=2))


## 5. Validate image paths and benchmark inputs
Check all referenced SfM and benchmark paths, ranking indices and nonempty query crop overlap; fully decode images on CPU before GPU work. This adds startup time but exposes missing or corrupt files before mining. No image content checksum is claimed.


In [ ]:
for name, key, expected in [
    ("oxford5k", "oxford", 5063),
    ("paris6k", "paris", 6392)
]:
    gp = choose(
        key + "_ground_truth",
        file_index.get("gnd_" + name + ".pkl", [])
    )

    with gp.open("rb") as f:
        gt = pickle.load(f)

    print(f"\n{name.upper()}")
    print("Metadata:", gp)
    print("Expected images:", expected)
    print("Actual images:", len(gt["imlist"]))
    print("Unique images:", len(set(gt["imlist"])))
    print("Queries:", len(gt["qimlist"]))
    print("Ground truth entries:", len(gt["gnd"]))

In [ ]:

# ============================================================
# CELL 11 — SfM subset-100, benchmarks, offline scaffold
# ============================================================

from pathlib import Path
import pickle
import hashlib
import json

# ------------------------------------------------------------
# 0. Explicit paths — independent of old Editor variables
# ------------------------------------------------------------

# Reuse the validated metadata/weights and honor PATH_OVERRIDES.
DATA_ROOT = WORKSPACE / 'data'
SFM_PKL = Path(SFM_PKL).resolve()
SFM_IMAGES = input_path(PATH_OVERRIDES['sfm_images'] or (SOURCE_SFM_PKL.parent / 'ims'))
WEIGHTS = Path(WEIGHTS).resolve()

assert SFM_PKL.is_file(), f"Missing metadata: {SFM_PKL}"
assert SFM_IMAGES.is_dir(), f"Missing images: {SFM_IMAGES}"
assert WEIGHTS.is_file(), f"Missing weights: {WEIGHTS}"

# ------------------------------------------------------------
# 1. Validate generated SfM subset-100
# ------------------------------------------------------------

with SFM_PKL.open("rb") as f:
    active_sfm_db = pickle.load(f)

train = active_sfm_db["train"]
val = active_sfm_db["val"]

assert len(train["cids"]) == 15953
assert len(set(train["cluster"])) == 100
assert len(train["qidxs"]) == 34337
assert len(train["pidxs"]) == 34337

assert len(val["cids"]) == 739
assert len(set(val["cluster"])) == 20
assert len(val["qidxs"]) == 181
assert len(val["pidxs"]) == 181

for partition_name, partition in [
    ("train", train),
    ("val", val)
]:
    n = len(partition["cids"])
    clusters = partition["cluster"]

    for q, p in zip(
        partition["qidxs"],
        partition["pidxs"]
    ):
        q, p = int(q), int(p)

        assert 0 <= q < n
        assert 0 <= p < n
        assert q != p
        assert clusters[q] == clusters[p]

    print(
        f"PASS {partition_name}: "
        f"{n} images, "
        f"{len(set(clusters))} clusters, "
        f"{len(partition['qidxs'])} pairs"
    )

# Check the exact CID layout used by the original loader, before any GPU mining.
referenced = [Path(datahelpers.cid2filename(cid, str(SFM_IMAGES)))
              for part in ['train', 'val'] for cid in active_sfm_db[part]['cids']]
from concurrent.futures import ThreadPoolExecutor
import stat as file_stat
PREPARATION_IO_WORKERS = 16

def _stat_sfm_image(path):
    try:
        info = path.stat()
    except FileNotFoundError:
        return path, None
    return path, info.st_size if file_stat.S_ISREG(info.st_mode) else None

stage('Checking SfM paths with bounded parallel I/O')
with ThreadPoolExecutor(max_workers=PREPARATION_IO_WORKERS) as executor:
    stats = list(executor.map(_stat_sfm_image, referenced))
missing_images = [str(path) for path,size in stats if size is None]
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} SfM images missing; first paths: {missing_images[:10]}')
SFM_IMAGE_BYTES = {str(path): size for path,size in stats}
print('SfM image paths checked:',len(SFM_IMAGE_BYTES))

# ------------------------------------------------------------
# 2. Initialize benchmarks explicitly
# ------------------------------------------------------------

BENCHMARKS = {}

for name, expected_images in [
    ("oxford5k", 5063),
    ("paris6k", 6392)
]:
    key = 'oxford' if name == 'oxford5k' else 'paris'
    gp = choose(key + '_ground_truth', file_index.get('gnd_' + name + '.pkl', []))
    jpg = input_path(PATH_OVERRIDES[key + '_images'] or (gp.parent / 'jpg'))

    assert gp.is_file(), f"Missing ground truth: {gp}"
    assert jpg.is_dir(), f"Missing image directory: {jpg}"

    with gp.open("rb") as f:
        gt = pickle.load(f)

    assert all(
        k in gt for k in ("imlist", "qimlist", "gnd")
    ), name

    assert len(gt["imlist"]) == expected_images
    assert len(set(gt["imlist"])) == expected_images
    assert len(gt["qimlist"]) == 55
    assert len(gt["gnd"]) == 55

    # Check every gallery and query image exists.
    all_ids = set(gt["imlist"]) | set(gt["qimlist"])

    for image_id in all_ids:
        assert isinstance(image_id,str) and Path(image_id).name == image_id
    benchmark_paths = [jpg / f'{image_id}.jpg' for image_id in sorted(all_ids)]
    with ThreadPoolExecutor(max_workers=PREPARATION_IO_WORKERS) as executor:
        present = list(executor.map(Path.is_file, benchmark_paths))
    missing = [str(path) for path,ok in zip(benchmark_paths,present) if not ok]
    if missing:
        raise FileNotFoundError(f'{name}: missing benchmark files: {missing[:10]}')

    # Invalid ranking indices or empty query crops otherwise fail much later.
    for query_id, ground_truth in zip(gt['qimlist'], gt['gnd']):
        for category in ['ok', 'junk']:
            indices = ground_truth.get(category, [])
            assert all(isinstance(i, (int, np.integer)) and 0 <= i < expected_images for i in indices), (name, category)
        bbox = ground_truth['bbx']
        assert len(bbox) == 4 and all(math.isfinite(float(v)) for v in bbox), (name, query_id)
        x0, y0, x1, y1 = map(float, bbox)
        with Image.open(jpg / f'{query_id}.jpg') as image:
            w, h = image.size
        assert x0 < x1 and y0 < y1 and x1 > 0 and y1 > 0 and x0 < w and y0 < h, (name, query_id, bbox)

    BENCHMARKS[name] = {
        "ground_truth": gp,
        "images": jpg,
        "gnd": gt
    }

    print(
        f"PASS {name}: "
        f"{expected_images} gallery images, "
        f"55 queries"
    )

# Full decoding catches truncated/corrupt images before training.
image_paths = list(dict.fromkeys(referenced + [b['images'] / (cid + '.jpg')
    for b in BENCHMARKS.values() for cid in set(b['gnd']['imlist']) | set(b['gnd']['qimlist'])]))
stage(f'Fully decoding {len(image_paths)} images with bounded parallel I/O')
def _decode_preparation_image(path):
    try:
        with Image.open(path) as image:
            image.convert('RGB').load()
    except Exception as error:
        raise RuntimeError(f'Image cannot be decoded: {path}: {error}') from error
    return None

with ThreadPoolExecutor(max_workers=PREPARATION_IO_WORKERS) as executor:
    for index,_ in enumerate(executor.map(_decode_preparation_image,image_paths),1):
        if index % 1000 == 0 or index == len(image_paths):
            print(f'Image decode preflight: {index}/{len(image_paths)}',flush=True)


In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor

EXPORT = WORKING / 'b0_sfm100_prepared'
if EXPORT.exists():
    raise RuntimeError('Export folder already exists. Use a fresh session or choose another EXPORT; no files are overwritten.')
EXPORT.mkdir()
needed_bytes = sum(SFM_IMAGE_BYTES[str(path)] for path in referenced)
free_bytes = shutil.disk_usage(EXPORT).free
if needed_bytes + 2 * 1024**3 > free_bytes:
    raise RuntimeError(f'Insufficient export space: need image bytes={needed_bytes} plus 2 GiB reserve; free={free_bytes}')
for source in [SFM_PKL, FILTER_PATH, PROVENANCE_PATH]:
    shutil.copyfile(source, EXPORT / source.name)

def copy_verified_image(source):
    relative = source.relative_to(SFM_IMAGES)
    destination = EXPORT / 'ims' / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    size = 0
    with source.open('rb') as inp, destination.open('xb') as output:
        for chunk in iter(lambda: inp.read(1024*1024), b''):
            output.write(chunk)
            digest.update(chunk)
            size += len(chunk)
    return relative.as_posix(), dict(bytes=size, sha256=digest.hexdigest())

stage(f'Exporting {len(referenced)} original SfM images ({needed_bytes/1e9:.2f} GB)')
with ThreadPoolExecutor(max_workers=8) as executor:
    exported = dict(executor.map(copy_verified_image, referenced))
manifest = dict(format='b0_prepared_subset_v1', dataset_identity=DATASET_IDENTITY,
                metadata_files={p.name: sha256_file(EXPORT/p.name) for p in [SFM_PKL,FILTER_PATH,PROVENANCE_PATH]},
                sfm_images=exported, full_image_decode_preflight=True,
                note='Original image bytes; no resize/recompression. Training samples hashes and decodes; lazy loading detects other failures.')
manifest_path = EXPORT / 'prepared_manifest.json'
manifest_path.write_text(json.dumps(manifest,sort_keys=True,indent=2))
print('PREPARATION COMPLETE:', EXPORT)
print('Publish this folder as a Kaggle Dataset, then attach it to the training notebook.')
_session_log_stop()
